In [2]:
import pandas as pd

In [5]:
df = pd.read_csv("/content/Cleaned_Amazon_Reviews.csv")

In [6]:
  df

,Id,ProductId,UserId,ProfileName,HelpfulnessNumerator,HelpfulnessDenominator,Score,Time,Summary,Text,Helpfulness_Ratio,Review_Length
0,1,B001E4KFG0,A3SGXH7AUHU8GW,delmartian,1,1,5,1970-01-01 00:00:01.303862400,good quality dog food,bought several vitality canned dog food produc...,1.000000,23.0
1,2,B00813GRG4,A1D87F6ZCVE5NK,dll pa,0,0,1,1970-01-01 00:00:01.346976000,advertised,product arrived labeled jumbo salted peanutsth...,0.000000,18.0
2,3,B000LQOCH0,ABXLMWJIXXAIN,"Natalia Corres ""Natalia Corres""",1,1,4,1970-01-01 00:00:01.219017600,delight say,confection around century light pillowy citrus...,1.000000,40.0
3,4,B000UA0QIQ,A395BORC6FGVXV,Karl,3,3,2,1970-01-01 00:00:01.307923200,cough medicine,looking secret ingredient robitussin believe f...,1.000000,18.0
4,5,B006K2ZZ7K,A1UQRSCLF8GW1T,"Michael D. Bigham ""M. Wassir""",0,0,5,1970-01-01 00:00:01.350777600,great taffy,great taffy great price wide assortment yummy ...,0.000000,13.0
...,...,...,...,...,...,...,...,...,...,...,...,...
40304,40309,B005Q1812W,AOFMKFQLJCAK9,Reader,0,0,4,1970-01-01 00:00:01.348531200,excellent product expensive,like others posted kale krunch chip think wond...,0.000000,19.0
40305,40310,B005Q1812W,A16W3FMQSX1OUB,"Cesaco ""Chelsea""",0,0,5,1970-01-01 00:00:01.338076800,love,addicted kale chip first time tried sure reall...,0.000000,21.0
40306,40311,B006DQZ064,A224KF8WZ0KIWR,lytgreenmom,9,9,4,1970-01-01 00:00:01.329177600,like,friend told new capsule available fit nespress...,1.000000,37.0
40307,40312,B006DQZ064,A2TO2BN3P4C00L,Music Fan Jeff,12,13,3,1970-01-01 00:00:01.329004800,good nespressos darker capsule,tried variety ecc capsule citiz machine far su...,0.923077,162.0


In [8]:
X = df['Text']
y = df['Score']
print(X)
print(y)

0        bought several vitality canned dog food produc...
1        product arrived labeled jumbo salted peanutsth...
2        confection around century light pillowy citrus...
3        looking secret ingredient robitussin believe f...
4        great taffy great price wide assortment yummy ...
                               ...                        
40304    like others posted kale krunch chip think wond...
40305    addicted kale chip first time tried sure reall...
40306    friend told new capsule available fit nespress...
40307    tried variety ecc capsule citiz machine far su...
40308    bought package first came initial reaction pri...
Name: Text, Length: 40309, dtype: object
0        5
1        1
2        4
3        2
4        5
        ..
40304    4
40305    5
40306    4
40307    3
40308    3
Name: Score, Length: 40309, dtype: int64


In [9]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print(X_train.shape)
print(X_test.shape)

(32247,)
(8062,)


## Feature Engineering 1: Bag of Words (CountVectorizer)

In [10]:
from sklearn.feature_extraction.text import CountVectorizer

count_vectorizer = CountVectorizer(
    max_features=5000
)

X_train_bow = count_vectorizer.fit_transform(X_train)

X_test_bow = count_vectorizer.transform(X_test)

print("Training Shape:", X_train_bow.shape)
print("Testing Shape:", X_test_bow.shape)

Training Shape: (32247, 5000)
Testing Shape: (8062, 5000)


In [11]:
print(count_vectorizer.get_feature_names_out()[:20])

['ability' 'able' 'absolute' 'absolutely' 'absorb' 'absorbed' 'abundant'
 'acai' 'accept' 'acceptable' 'accepted' 'access' 'accident'
 'accidentally' 'according' 'account' 'accurate' 'accustomed' 'acerola'
 'ache']


## Feature Engineering 2: TF-IDF Vectorizer

In [12]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf_vectorizer = TfidfVectorizer(
    max_features=5000
)

X_train_tfidf = tfidf_vectorizer.fit_transform(X_train)

X_test_tfidf = tfidf_vectorizer.transform(X_test)

print("Training Shape:", X_train_tfidf.shape)
print("Testing Shape:", X_test_tfidf.shape)

print(tfidf_vectorizer.get_feature_names_out()[:20])

Training Shape: (32247, 5000)
Testing Shape: (8062, 5000)
['ability' 'able' 'absolute' 'absolutely' 'absorb' 'absorbed' 'abundant'
 'acai' 'accept' 'acceptable' 'accepted' 'access' 'accident'
 'accidentally' 'according' 'account' 'accurate' 'accustomed' 'acerola'
 'ache']


## Train and Compare Multiple Models

In [13]:
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)

In [14]:
def evaluate_model(model, X_train, X_test, y_train, y_test):

    # Train the model
    model.fit(X_train, y_train)

    # Predictions
    y_pred = model.predict(X_test)

    # Metrics
    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred, average='weighted')
    recall = recall_score(y_test, y_pred, average='weighted')
    f1 = f1_score(y_test, y_pred, average='weighted')

    print("="*60)
    print(model.__class__.__name__)
    print("="*60)

    print("Accuracy :", accuracy)
    print("Precision:", precision)
    print("Recall   :", recall)
    print("F1 Score :", f1)

    print("\nConfusion Matrix")
    print(confusion_matrix(y_test, y_pred))

    print("\nClassification Report")
    print(classification_report(y_test, y_pred))

    return accuracy, precision, recall, f1

In [18]:
evaluate_model(MultinomialNB(), X_train_bow, X_test_bow, y_train, y_test)

MultinomialNB
Accuracy : 0.6592656908955594
Precision: 0.6434708432466973
Recall   : 0.6592656908955594
F1 Score : 0.6496188609216275

Confusion Matrix
[[ 457   96   60   32  119]
 [ 128   97   86   41  103]
 [  91   73  172  113  192]
 [  59   54  133  318  585]
 [ 194   54  171  363 4271]]

Classification Report
              precision    recall  f1-score   support

           1       0.49      0.60      0.54       764
           2       0.26      0.21      0.23       455
           3       0.28      0.27      0.27       641
           4       0.37      0.28      0.32      1149
           5       0.81      0.85      0.83      5053

    accuracy                           0.66      8062
   macro avg       0.44      0.44      0.44      8062
weighted avg       0.64      0.66      0.65      8062



(0.6592656908955594,
 0.6434708432466973,
 0.6592656908955594,
 0.6496188609216275)

In [17]:
evaluate_model(LogisticRegression(), X_train_bow, X_test_bow, y_train, y_test)

LogisticRegression
Accuracy : 0.689034978913421
Precision: 0.651666871169004
Recall   : 0.689034978913421
F1 Score : 0.6650444094230696

Confusion Matrix
[[ 448   87   43   31  155]
 [ 104   99   84   46  122]
 [  56   80  158  103  244]
 [  36   43   85  290  695]
 [  84   44   84  281 4560]]

Classification Report
              precision    recall  f1-score   support

           1       0.62      0.59      0.60       764
           2       0.28      0.22      0.25       455
           3       0.35      0.25      0.29       641
           4       0.39      0.25      0.31      1149
           5       0.79      0.90      0.84      5053

    accuracy                           0.69      8062
   macro avg       0.48      0.44      0.46      8062
weighted avg       0.65      0.69      0.67      8062



/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


(0.689034978913421, 0.651666871169004, 0.689034978913421, 0.6650444094230696)

In [19]:
evaluate_model(LinearSVC(), X_train_bow, X_test_bow, y_train, y_test)

LinearSVC
Accuracy : 0.6766311089059787
Precision: 0.6405746046179055
Recall   : 0.6766311089059787
F1 Score : 0.6533158130486809

Confusion Matrix
[[ 413   92   52   32  175]
 [ 103  110   74   49  119]
 [  69   71  162   94  245]
 [  48   60   87  260  694]
 [ 108   71  103  261 4510]]

Classification Report
              precision    recall  f1-score   support

           1       0.56      0.54      0.55       764
           2       0.27      0.24      0.26       455
           3       0.34      0.25      0.29       641
           4       0.37      0.23      0.28      1149
           5       0.79      0.89      0.84      5053

    accuracy                           0.68      8062
   macro avg       0.47      0.43      0.44      8062
weighted avg       0.64      0.68      0.65      8062



(0.6766311089059787,
 0.6405746046179055,
 0.6766311089059787,
 0.6533158130486809)

In [20]:
evaluate_model(DecisionTreeClassifier(), X_train_bow, X_test_bow, y_train, y_test)

DecisionTreeClassifier
Accuracy : 0.603448275862069
Precision: 0.5863451594751574
Recall   : 0.603448275862069
F1 Score : 0.5940638220270595

Confusion Matrix
[[ 297   50   78   69  270]
 [  97   84   66   52  156]
 [  69   46  154   87  285]
 [  87   47   97  309  609]
 [ 226  116  208  482 4021]]

Classification Report
              precision    recall  f1-score   support

           1       0.38      0.39      0.39       764
           2       0.24      0.18      0.21       455
           3       0.26      0.24      0.25       641
           4       0.31      0.27      0.29      1149
           5       0.75      0.80      0.77      5053

    accuracy                           0.60      8062
   macro avg       0.39      0.38      0.38      8062
weighted avg       0.59      0.60      0.59      8062



(0.603448275862069, 0.5863451594751574, 0.603448275862069, 0.5940638220270595)

In [21]:
evaluate_model(RandomForestClassifier(), X_train_bow, X_test_bow, y_train, y_test)

RandomForestClassifier
Accuracy : 0.6957330687174399
Precision: 0.7280870819573311
Recall   : 0.6957330687174399
F1 Score : 0.6222554067003829

Confusion Matrix
[[ 279    4    6    6  469]
 [  33   53    4   14  351]
 [  17    2   88   23  511]
 [   8    1    3  176  961]
 [  22    0    3   15 5013]]

Classification Report
              precision    recall  f1-score   support

           1       0.78      0.37      0.50       764
           2       0.88      0.12      0.21       455
           3       0.85      0.14      0.24       641
           4       0.75      0.15      0.25      1149
           5       0.69      0.99      0.81      5053

    accuracy                           0.70      8062
   macro avg       0.79      0.35      0.40      8062
weighted avg       0.73      0.70      0.62      8062



(0.6957330687174399,
 0.7280870819573311,
 0.6957330687174399,
 0.6222554067003829)

In [22]:
evaluate_model(KNeighborsClassifier(), X_train_bow, X_test_bow, y_train, y_test)

KNeighborsClassifier
Accuracy : 0.6126271396675763
Precision: 0.5215570237769507
Recall   : 0.6126271396675763
F1 Score : 0.543159263468832

Confusion Matrix
[[ 113   40   25   43  543]
 [  43   34   23   38  317]
 [  39   26   59   61  456]
 [  43   26   45  117  918]
 [  89   69   83  196 4616]]

Classification Report
              precision    recall  f1-score   support

           1       0.35      0.15      0.21       764
           2       0.17      0.07      0.10       455
           3       0.25      0.09      0.13       641
           4       0.26      0.10      0.15      1149
           5       0.67      0.91      0.78      5053

    accuracy                           0.61      8062
   macro avg       0.34      0.27      0.27      8062
weighted avg       0.52      0.61      0.54      8062



(0.6126271396675763, 0.5215570237769507, 0.6126271396675763, 0.543159263468832)